# EDA sơ bộ — Glints & JobsGO

Notebook này đọc **raw detail snapshots thật trong MinIO**, chọn version mới nhất cho mỗi tin, tạo bảng kiểm kê và các biểu đồ theo kỹ năng `.agents/skills/eda-intelligence/SKILL.md`. Nó không đọc synthetic sample fixtures và không tự bịa số liệu.

**Các nhóm EDA:** missing value/field inventory; phân phối lương; quan hệ kinh nghiệm–lương; dispersion/outlier IQR; tương quan Pearson và Spearman; tần suất kỹ năng + Pareto. Lương chỉ đem so sánh khi có căn cứ về currency và kỳ trả lương. USD conversion bị tắt nếu không khai báo `FX_VND_PER_USD` trong `.env`.

Nếu JobsGO không có detail snapshot (ví dụ crawler nhận HTTP 403), nguồn đó hiện `0 snapshots` / `N/A`; không thể kết luận độ phủ hay so sánh công bằng với Glints cho đến khi xử lý blocker truy cập hợp lệ.


In [ ]:
from pathlib import Path
import os, sys

# Hỗ trợ chạy notebook từ cả thư mục project root và notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "run.py").is_file():
    if (PROJECT_ROOT.parent / "run.py").is_file():
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise FileNotFoundError("Không tìm thấy run.py. Mở notebook trong project VN-IT-Job-Mining.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env", override=False)
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from scripts.run_glints_jobsgo_eda import run_eda

print("Project:", PROJECT_ROOT)
print("EDA output:", PROJECT_ROOT / "reports" / "generated")
print("MinIO endpoint:", os.getenv("S3_ENDPOINT_URL") or os.getenv("MINIO_ENDPOINT", "http://localhost:9000"))


## 1. Đọc MinIO, chốt version mới nhất và tạo output

Cell sau sẽ kiểm tra kết nối S3, tải các object khớp mẫu `raw/<source>/YYYY-MM-DD/<job_id>_vN.json`, loại listing objects/fixtures khỏi tập phân tích, tính các bảng thống kê và lưu báo cáo/biểu đồ trong `reports/generated/`. Nó cũng cập nhật các section có marker trong hai báo cáo yêu cầu của dự án.


In [ ]:
results = run_eda()
summary_by_source = results["summary"]
missing_rate_table = results["missing_rate"]
iqr_summary = results["iqr"]
skills_frequency = results["skills"]
raw_field_inventory = results["inventory"]
df = results["df"]
read_errors = pd.DataFrame(results["errors"])

print("Latest detail snapshots:", len(df))
display(summary_by_source)


## 2. Kiểm kê trường và Missing Rate

`missing_rate_table` phân biệt chuỗi rỗng/null/field thiếu theo từng trường nghiệp vụ; `raw_field_inventory` phân biệt `presence_rate_pct`, `non_null_rate_pct` và `absent_rate_pct` của trường JSON listing. HTML selector presence là chỉ dấu kiểm kê cấu trúc DOM, không đồng nghĩa selector có ý nghĩa nghiệp vụ hoặc extractor đã parse đúng.


In [ ]:
display(missing_rate_table)
display(raw_field_inventory.head(100))
if not read_errors.empty:
    print("Có object không parse được; xem file read_errors CSV.")
    display(read_errors.head(30))
else:
    print("Không phát hiện object detail nào bị lỗi đọc/JSON parse.")


## 3. Lương, ngoại lai và so sánh giữa nguồn

- `salary_public_numeric` nghĩa là đã tìm thấy cận lương số ở đơn vị gốc; không khẳng định luôn quy đổi được sang tháng.
- `salary_comparable_monthly` chỉ đúng khi currency và period đủ rõ để so sánh.
- IQR dùng cận `max(0, Q1 - 1.5×IQR)` và `Q3 + 1.5×IQR`. Ngoại lai là tín hiệu cần kiểm tra HTML/raw gốc, không phải lý do tự động xóa tin.


In [ ]:
display(iqr_summary)
if not df.empty:
    display(df[["source", "title", "company", "salary_text", "salary_currency", "salary_period",
                "salary_min_million_vnd_month", "salary_max_million_vnd_month",
                "min_experience_years", "salary_comparable_monthly"]].head(30))


## 4. Từ khóa kỹ năng và Pareto

Đây là heuristic scan với ranh giới từ (word boundaries) nhằm giảm false positive, ví dụ không đếm `Go` trong `Google`/`MongoDB`. Đây vẫn chưa phải taxonomy kỹ năng có kiểm duyệt; xem `skills_keyword_scan` như tín hiệu khám phá chứ không phải trường tags nguyên gốc của JobsGO.


In [ ]:
display(skills_frequency.head(30))


## 5. Năm nhóm biểu đồ và file xuất

Bộ chart gồm: (1) distribution histogram lương tháng có thể so sánh; (2) relationship kinh nghiệm–lương; (3) dispersion boxplot theo nguồn cùng IQR; (4) correlation matrices Pearson/Spearman; (5) categorical frequency + Pareto kỹ năng. Nếu dữ liệu thực chưa đủ, biểu đồ ghi rõ thiếu dữ liệu thay vì thêm quan sát giả. Hai chart chất lượng dữ liệu bổ sung missing rate và tỷ lệ lương số theo nguồn.


In [ ]:
figures_dir = results["figures_dir"]
for image_path in sorted(figures_dir.glob("*.png")):
    print(image_path.name)
    display(Image(filename=str(image_path)))


## 6. Vị trí output và cách kết luận

Script đã tạo các bảng CSV theo ngày trong `reports/generated/`, các PNG trong `reports/generated/figures/`, và một file Markdown `glints_jobsgo_eda_summary_YYYY-MM-DD.md`. Nó cập nhật section `AUTO-GENERATED EDA RESULTS` của `reports/glints_jobsgo_eda.md` và section `AUTO-GENERATED MEASURED INVENTORY` trong `reports/glints_jobsgo_inventory.md`.

Khi viết kết luận: chỉ so sánh Glints–JobsGO nếu cả hai có đủ detail snapshot và các trường/đơn vị tương thích. Ghi rõ ngày chạy, số URL trong catalog, số latest snapshot, số lỗi đọc và tỷ lệ missing. Không biến một lượt crawl thành công thành tuyên bố đã thu thập toàn bộ website. Nếu EDA không có dữ liệu, cần chạy discovery/detail hoặc giải quyết blocker truy cập trước rồi **Run All** lại.
